# 1. Preprocessing Raw Data

The raw inputs used by this notebook are not included. The preprocessed output, `01_preprocess.h5`, is supplied in the corresponding data folder. Start with `02_denoise.ipynb` to run the analysis from that data.

In [ ]:
# Main imports
import sys
from pathlib import Path

# Additional imports
from scipy.io import loadmat
from natsort import natsorted

# Add the project root to the Python path
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

# Import custom modules
from src.processing import tangle_data_load, tangle_data_standardize_save
from src.config import PROJECT_ROOT, DATA_ROOT, initialize_h5
from src.utilities import repack_h5, inspect_h5

In [ ]:
# Directory and files setup

TYPE = 'tangle_01'
STAGE = '01_preprocess'

INPUT_DIR = DATA_ROOT / TYPE / 'raw'
OUTPUT_PATH = DATA_ROOT / TYPE / (STAGE + '.h5')

print('INPUT_DIR: \t{:}'.format(INPUT_DIR))
print('OUTPUT_PATH: \t{:}'.format(OUTPUT_PATH))
print()

# Check raw inputs before initializing the output file.
required_raw_files = ('lengthscales.txt', 'piston_data.mat', 'piston_ref.mat', 'data.mat')
missing_raw_files = [name for name in required_raw_files if not (INPUT_DIR / name).is_file()]
if missing_raw_files:
    raise FileNotFoundError(
        f"Missing raw input files: {', '.join(missing_raw_files)}. "
        "Use the supplied 01_preprocess.h5 and start with 02_denoise.ipynb."
    )

# Initialize output file
overwrite = True
verbose = True

initialize_h5(OUTPUT_PATH, overwrite=overwrite, verbose=verbose)

## Preprocess Data

- Data structure: `video_results = mat_data['videoResults1']` instead of `'videoResults'`
- Frequency terminology: Added `freq_part = freq_part.replace('moderate', 'opt')` normalizes naming convention consistently
- Missing `piston_left` handling: `piston_left = piston_data['cos_sig']` for backward compatibility

### Experiment parameters, piston trajectories, annotations

In [ ]:
# Load raw data files

# Load length scales and create experiment parameter dictionary
exp_params = {}
exp_name_mapping = {}

length_scale_file = INPUT_DIR / 'lengthscales.txt'
with open(length_scale_file, 'r') as file:
    lines = natsorted(file.readlines()[1:])
    for line in lines:
        line = line.strip().split('\t')
        original_name = line[0].replace('.mp4','')
        length_scales = [float(line[1]), float(line[2])]
        
        # Parse original name to extract parameters
        parts = original_name.split('_')
        tw_part = parts[0]  # e.g., "0TW", "2TW", "10TW"
        freq_part = parts[1]  # e.g., "low", "opt", "high"
        freq_part = freq_part.replace('moderate', 'opt')  # Normalize 'moderate' to 'opt'

        tumbling_weights = int(tw_part.replace('TW', ''))
        
        # Create new clean name
        new_name = f"tw{tumbling_weights}_{freq_part}"
        
        # Store everything in unified dictionary
        exp_params[new_name] = {
            'original_name': original_name,
            'tumbling_weights': tumbling_weights,
            'frequency_type': freq_part,
            'length_scale_left': length_scales[0],
            'length_scale_right': length_scales[1]
        }
        exp_name_mapping[original_name] = new_name

print(f"Found {len(exp_params)} experiments:")
if verbose:
    for new_name, data in exp_params.items():
        print(f"  {data['original_name'].ljust(20)} -> {new_name.ljust(10)} (TW={data['tumbling_weights']}, freq={data['frequency_type']}, scales={data['length_scale_left']:.2f}(L), {data['length_scale_right']:.2f}(R))")
print()

In [ ]:
# Piston data
piston_file = INPUT_DIR / 'piston_data.mat'
piston_data = loadmat(str(piston_file))
cos_sig = piston_data['cos_sig']
piston_left = piston_data['cos_sig'] #piston_data['piston_left']
# piston_left is not included in this dataset, but defined for backward compatibility

piston_ref_file = INPUT_DIR / 'piston_ref.mat'
piston_ref = loadmat(str(piston_ref_file))['ref_pts']

print(f"Loaded piston data: cos_sig shape {cos_sig.shape}, piston_left shape {piston_left.shape}")
print(f"Loaded piston reference points: shape {piston_ref.shape}")

# Reshape cos_sig to remove unnecessary dimensions
for n_exp in range(piston_data['cos_sig'].shape[1]):
    piston_data['cos_sig'][0][n_exp] = piston_data['cos_sig'][0][n_exp].T

# Parameters for piston
PISTON_AMPLITUDE_SCALING = 15.0/2  # mm, experimentally determined
frequency_mapping = {
    'low': 4,
    'opt': 17,
    'high': 37
}   # frequency in Hz

In [ ]:
# Annotations

exp_annotations = {
    'tw0_low': {
        'extracted': False,
        'data_quality': 'good',
        'comments': 'steady length, pendulum mode, good dynamics',
        'time_range': (0, 100)  # (start, end) in seconds
    },
    'tw0_opt': {
        'extracted': False,
        'data_quality': 'bad',
        'comments': 'string lengthening (partial extraction)',
        'time_range': (0, 100)
    },
    'tw0_high': {
        'extracted': True,
        'data_quality': 'bad',
        'comments': 'extraction, clear string lengthening',
        'time_range': (0,100)
    },
    'tw2_low': {
        'extracted': False,
        'data_quality': 'bad',
        'comments': 'steady length',
        'time_range': (0, 100)
    },
    'tw2_opt': {
        'extracted': False,
        'data_quality': 'good',
        'comments': 'string lengthening (partial extraction)',
        'time_range': (0, 100)
    },
    'tw2_high': {
        'extracted': False,
        'data_quality': 'bad',
        'comments': 'string lengthening (partial extraction)',
        'time_range': (0, 100)
    },
    'tw10_low': {
        'extracted': False,
        'data_quality': 'bad',
        'comments': 'steady length',
        'time_range': (0, 100)
    },
    'tw10_opt': {
        'extracted': False,
        'data_quality': 'good',
        'comments': 'string lengthening (partial extraction)',
        'time_range': (0, 100)
    },
    'tw10_high': {
        'extracted': False,
        'data_quality': 'bad',
        'comments': 'string lengthening (partial extraction)',
        'time_range': (0, 100)
    }
}

# Save to file
import json
annotations_file = INPUT_DIR / 'exp_annotations.json'
with open(annotations_file, 'w') as f:
    json.dump(exp_annotations, f, indent=2)
print(f"Saved annotations to {annotations_file}")

### Main experimental data

In [ ]:
# Preprocess main experimental data

data_file=INPUT_DIR / 'data.mat'
video_results_key='videoResults1'  # 'videoResults' for tangle_00, 'videoResults1' for tangle_01

hdf5_options={'compression': 'gzip', 'compression_opts': 4}

# Load main experimental data from MATLAB file
processed_datasets = tangle_data_load(
    data_file=data_file,
    exp_name_mapping=exp_name_mapping,
    exp_params=exp_params,
    cos_sig=cos_sig,
    piston_left=piston_left,
    piston_ref=piston_ref,
    video_results_key=video_results_key,
    verbose=verbose
)

# Convert to standardized format and save
tangle_data_standardize_save(
    processed_datasets=processed_datasets,
    exp_annotations=exp_annotations,
    frequency_mapping=frequency_mapping,
    piston_amplitude_scaling=PISTON_AMPLITUDE_SCALING,
    output_path=OUTPUT_PATH,
    hdf5_options=hdf5_options,
    verbose=verbose
)

# Repack and inspect
repack_h5(OUTPUT_PATH, in_place=True)
inspect_h5(OUTPUT_PATH)